<p align="center">
  <h1 align="center">🍳 Cookbook 02: ML Feature Engineering & VIF</h1>
  <p align="center">
    <strong>GradTracer for XGBoost, LightGBM, and Tabular Feature Selection</strong>
  </p>
</p>

---

While PyTorch FlowTracker analyzes backpropagation, GradTracer also supports **Tree Ensembles (XGBoost/LightGBM)** through the `FeatureAnalyzer` and `TreeDynamicsTracker`.

This recipe has been updated to support multiple tabular datasets and both XGBoost and LightGBM comparison.

## 1. Setup & Multi-Dataset Configuration

In [ ]:
# !pip install gradtracer xgboost lightgbm scikit-learn pandas statsmodels

In [ ]:
import xgboost as xgb
import lightgbm as lgb
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_california_housing, load_breast_cancer, load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, accuracy_score
from gradtracer import FeatureAnalyzer, TreeDynamicsTracker

# --- CONFIGURATION ---
BENCHMARK = {
    "dataset": "california", # Options: "california", "diabetes", "cancer"
    "model_type": "xgboost"   # Options: "xgboost", "lightgbm"
}

def load_data(name):
    if name == "california":
        data = fetch_california_housing()
        return pd.DataFrame(data.data, columns=data.feature_names), data.target, "regression"
    elif name == "diabetes":
        data = load_diabetes()
        return pd.DataFrame(data.data, columns=data.feature_names), data.target, "regression"
    elif name == "cancer":
        data = load_breast_cancer()
        return pd.DataFrame(data.data, columns=data.feature_names), data.target, "classification"

X, y, task_type = load_data(BENCHMARK["dataset"])

# Introduce an artificial correlated feature to trigger the VIF warning
X['Redundant_Feature'] = X.iloc[:, 0] * 1.5 + np.random.normal(0, 0.05, len(X))

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Loaded {BENCHMARK['dataset']} dataset ({task_type}) with {X.shape[1]} features.")

## 2. Train Model

In [ ]:
print(f"Training {BENCHMARK['model_type'].upper()} Model...")

if BENCHMARK["model_type"] == "xgboost":
    params = {
        'objective': 'reg:squarederror' if task_type == "regression" else 'binary:logistic',
        'max_depth': 4, 'eta': 0.1, 'eval_metric': 'rmse' if task_type == "regression" else 'logloss'
    }
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dtest = xgb.DMatrix(X_test, label=y_test)
    bst = xgb.train(params, dtrain, num_boost_round=100, verbose_eval=False)
    preds = bst.predict(dtest)
    model_obj = bst
else:
    params = {
        'objective': 'regression' if task_type == "regression" else 'binary',
        'metric': 'rmse' if task_type == "regression" else 'binary_logloss',
        'verbosity': -1
    }
    train_data = lgb.Dataset(X_train, label=y_train)
    valid_data = lgb.Dataset(X_test, label=y_test, reference=train_data)
    bst = lgb.train(params, train_data, num_boost_round=100)
    preds = bst.predict(X_test)
    model_obj = bst

if task_type == "regression":
    score = np.sqrt(mean_squared_error(y_test, preds))
    print(f"✅ Initial RMSE: {score:.4f}")
else:
    score = accuracy_score(y_test, (preds > 0.5).astype(int))
    print(f"✅ Initial Accuracy: {score:.4f}")

## 3. Tree Dynamics Tracking & Feature Diagnosis

In [ ]:

tree_tracker = TreeDynamicsTracker(model_obj)
tree_tracker.report()

print("\n--- Running Feature Diagnosis ---")
analyzer = FeatureAnalyzer(model_obj, X_train, y_train, feature_names=X.columns.tolist())
vif_results = analyzer.multicollinearity(threshold=10.0)

for f in vif_results['warnings']:
    print(f"⚠️ Feature '{f['feature']}' has critical VIF ({f['vif']:.1f}).")

print("\n--- Top Feature Interactions ---")
interactions = analyzer.interactions(top_k=5)
for i, item in enumerate(interactions):
    print(f"{i+1}. {item['feat_a']} × {item['feat_b']} (Synergy Score: {item['synergy_score']:.4f})")

## 4. Pruning and Retraining Verification

In [ ]:
features_to_drop = [f['feature'] for f in vif_results['warnings']]
print(f"Pruning redundant features identified by GradTracer: {features_to_drop}")

X_train_clean = X_train.drop(columns=features_to_drop)
X_test_clean = X_test.drop(columns=features_to_drop)

if BENCHMARK["model_type"] == "xgboost":
    dtrain_clean = xgb.DMatrix(X_train_clean, label=y_train)
    dtest_clean = xgb.DMatrix(X_test_clean, label=y_test)
    bst_clean = xgb.train(params, dtrain_clean, num_boost_round=100, verbose_eval=False)
    preds_clean = bst_clean.predict(dtest_clean)
else:
    train_data_clean = lgb.Dataset(X_train_clean, label=y_train)
    bst_clean = lgb.train(params, train_data_clean, num_boost_round=100)
    preds_clean = bst_clean.predict(X_test_clean)

if task_type == "regression":
    score_clean = np.sqrt(mean_squared_error(y_test, preds_clean))
    improved = "Improved" if score_clean < score else "Maintained"
    print(f"\n✅ Cleaned RMSE: {score_clean:.4f} ({improved} performance with {len(X_train_clean.columns)} features!)")
else:
    score_clean = accuracy_score(y_test, (preds_clean > 0.5).astype(int))
    improved = "Improved" if score_clean > score else "Maintained"
    print(f"\n✅ Cleaned Accuracy: {score_clean:.4f} ({improved} performance with {len(X_train_clean.columns)} features!)")